# IAN 603 · Module 1
# Understanding Data in Informatics

Understand what data represents before cleaning, transforming, querying, or modeling it.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module1.html


## Learning goals

Work through the examples in order. Focus on the meaning of each row, the reason for each SQL clause, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 1

Before writing complex SQL, ask what the data **means**.

| Question | Example |
|---|---|
| What does one row represent? | one encounter |
| Is a column an identifier or a measure? | patient_id vs. temperature |
| Is zero meaningful? | 0 ER visits can be real |
| Does NULL mean unknown or not applicable? | missing temperature |
| Is a value merely valid, or actually accurate? | a plausible temperature can still be wrong |

### Measurement matters

| Example | Useful interpretation |
|---|---|
| patient ID | nominal identifier |
| triage level | ordered category |
| ER visits | count with meaningful zero |
| Celsius temperature | numeric, but zero is not an absence of temperature |
| total charge | quantitative measure with meaningful zero |

The habit for this module is simple:

> **Understand the row, column meaning, missingness, and quality before analysis.**


# Teaching database

Run the next cell to create or reset the Module 1 teaching data. You can return to this cell whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 1\n-- Understanding Data in Informatics\n-- ============================================================\n-- Companion file for s_module1.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- PURPOSE\n-- Module 1 is about understanding what data means before trying\n-- to analyze it. This file gives a small relational dataset for\n-- practicing data types, measurement meaning, missingness,\n-- validity, uniqueness, and basic profiling.\n--\n-- KEY IDEA\n-- A PostgreSQL data type tells you how a value is stored.\n-- It does not by itself tell you what the value means.\n-- ============================================================\n\nDROP TABLE IF EXISTS encounters;\n\nCREATE TABLE encounters (\n    encounter_id BIGSERIAL PRIMARY KEY,\n    patient_id TEXT NOT NULL,\n    encounter_date DATE NOT NULL,\n    triage_level TEXT NOT NULL\n        CHECK (triage_level IN ('Non-urgent', 'Urgent', 'Emergent')),\n    temperature_c DOUBLE PRECISION\n        CHECK (temperature_c BETWEEN 30 AND 45),\n    er_visits INTEGER NOT NULL\n        CHECK (er_visits >= 0),\n    total_charge NUMERIC(12, 2)\n        CHECK (total_charge >= 0)\n);\n\nINSERT INTO encounters\n    (patient_id, encounter_date, triage_level,\n     temperature_c, er_visits, total_charge)\nVALUES\n    ('P1001', DATE '2026-09-01', 'Urgent',     38.2, 3, 125.00),\n    ('P1002', DATE '2026-09-01', 'Non-urgent',36.9, 0,  65.00),\n    ('P1003', DATE '2026-09-02', 'Emergent',   39.4, 5, 980.00),\n    ('P1004', DATE '2026-09-02', 'Urgent',     37.8, 1, 210.00),\n    ('P1005', DATE '2026-09-03', 'Non-urgent',37.0, 0,  80.00),\n    ('P1006', DATE '2026-09-03', 'Urgent',     NULL, 2, 340.00),\n    ('P1007', DATE '2026-09-04', 'Emergent',   40.1, 4, 1200.00),\n    ('P1008', DATE '2026-09-04', 'Urgent',     38.0, 1, NULL);"

execute_script(setup_sql)


# INSPECT THE DATA


## INSPECT THE DATA · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT *\nFROM encounters\nORDER BY encounter_id;"

display_query(query, title="INSPECT THE DATA · Step 1")


## INSPECT THE DATA · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- PostgreSQL storage types.\nSELECT column_name, data_type\nFROM information_schema.columns\nWHERE table_name = 'encounters'\nORDER BY ordinal_position;"

display_query(query, title="INSPECT THE DATA · Step 2")


# ASK WHAT EACH COLUMN MEANS


## ASK WHAT EACH COLUMN MEANS · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- patient_id is stored as text and behaves like an identifier.\n-- It is categorical/nominal even though it contains digits.\n--\n-- triage_level is ordered categorical data.\n--\n-- er_visits is a count. Zero has a meaningful interpretation.\n--\n-- temperature_c is numeric but Celsius does not have a true zero.\n--\n-- total_charge is a quantitative measure with a meaningful zero.\n\n-- Categories and their frequencies.\nSELECT triage_level, COUNT(*) AS encounters\nFROM encounters\nGROUP BY triage_level\nORDER BY encounters DESC, triage_level;"

display_query(query, title="ASK WHAT EACH COLUMN MEANS · Step 1")


## ASK WHAT EACH COLUMN MEANS · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Quantitative summaries.\nSELECT\n    COUNT(*) AS rows,\n    COUNT(temperature_c) AS recorded_temperatures,\n    ROUND(AVG(temperature_c)::numeric,2) AS avg_temperature_c,\n    MIN(er_visits) AS min_er_visits,\n    MAX(er_visits) AS max_er_visits,\n    ROUND(AVG(total_charge),2) AS avg_charge\nFROM encounters;"

display_query(query, title="ASK WHAT EACH COLUMN MEANS · Step 2")


# BASIC DATA QUALITY PROFILING


## BASIC DATA QUALITY PROFILING · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Missing values.\nSELECT\n    COUNT(*) FILTER (WHERE temperature_c IS NULL) AS missing_temperature,\n    COUNT(*) FILTER (WHERE total_charge IS NULL) AS missing_charge\nFROM encounters;"

display_query(query, title="BASIC DATA QUALITY PROFILING · Step 1")


## BASIC DATA QUALITY PROFILING · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Uniqueness of the encounter identifier.\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT encounter_id) AS distinct_encounter_ids\nFROM encounters;"

display_query(query, title="BASIC DATA QUALITY PROFILING · Step 2")


## BASIC DATA QUALITY PROFILING · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Patient identifiers can repeat across encounters in real data.\nSELECT patient_id, COUNT(*) AS encounter_count\nFROM encounters\nGROUP BY patient_id\nORDER BY encounter_count DESC, patient_id;"

display_query(query, title="BASIC DATA QUALITY PROFILING · Step 3")


## BASIC DATA QUALITY PROFILING · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Range checks provide evidence about plausibility.\nSELECT *\nFROM encounters\nWHERE temperature_c IS NOT NULL\n  AND (temperature_c < 35 OR temperature_c > 40)\nORDER BY temperature_c;"

display_query(query, title="BASIC DATA QUALITY PROFILING · Step 4")


# GRAIN


## GRAIN · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Grain means what one row represents.\n-- Here: one row = one encounter.\n--\n-- Always state grain before aggregation or joining.\n\nSELECT\n    encounter_date,\n    COUNT(*) AS encounter_count,\n    SUM(er_visits) AS total_prior_er_visits,\n    SUM(total_charge) AS total_charge\nFROM encounters\nGROUP BY encounter_date\nORDER BY encounter_date;"

display_query(query, title="GRAIN · Step 1")


# Practice

1. Which columns are identifiers, categories, counts, or measures?
2. Find all rows with missing numeric measurements.
3. Count encounters by triage level.
4. Compare average charge across triage levels.
5. Explain why averaging patient_id would be meaningless.
MODULE 1 CHECKLIST
[ ] I can state what one row represents.
[ ] I distinguish storage type from measurement meaning.
[ ] I inspect missingness and validity before analysis.
[ ] I know that identifiers are not quantitative measures.

Try each task before looking back at the earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# When ready:
# display_query(practice_query, title="Practice")


# Module 1 summary

Before moving on, make sure you can explain the main ideas in your own words and can describe the **grain** or row meaning of the important query results.

A good final check is:

1. What question did the query answer?
2. What did one input row represent?
3. What does one output row represent?
4. How did SQL change the data shape or meaning?
5. How would you validate that the result is correct?

## Next
Continue to **Module 2**.
